# One Health Disease Intelligence: Spatial Querying, GIS Payloads, and Bulletins

This notebook implements the final phase of our early warning disease intelligence workflow:
1. **Query our spatiotemporal databases** using localized query filters matching our dashboard (Region-Wise Single/All, District-Wise).
2. **Export standardized GeoJSON FeatureCollections** and spatial heatmap coordinate arrays to feed our GIS dashboards.
3. **Catalog and search Monthly Prediction Bulletins** (historical 2-month-ahead forecast files) using our document library engine.

In [ ]:
# 1. Initialize Python Path for local module imports
import os
import sys
import json
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

print("System path initialized successfully.")

## Step 2: Load the Processed Risk Assessments
We load our finalized, evaluated output database compiled during our risk analysis and modeling steps.

In [ ]:
risk_data_path = "../data/processed/risk_assessment_output.csv"

if os.path.exists(risk_data_path):
    df_risk = pd.read_csv(risk_data_path)
    print(f"[SUCCESS] Loaded evaluated risk matrix containing {len(df_risk)} records.")
else:
    print("[ERROR] Missing processed risk assessments. Run the 02_risk_analysis.ipynb notebook first.")

## Step 3: Test Localized Spatial Query Workflows
Using the `ForewarningQueryEngine`, we simulate user-driven query requests matching our dashboard's UI filter panels.

In [ ]:
from agents.forewarning_query_engine import ForewarningQueryEngine

# Initialize Query Engine
query_engine = ForewarningQueryEngine(df_risk)

# Workflow 1: Region-Wise Single Disease (e.g. FMD risk across South Ethiopia region)
south_ethiopia_fmd = query_engine.query_region_single_disease(
    region="South_Ethiopia", 
    disease_name="Foot and Mouth Disease"
)

# Workflow 2: Region-Wise All Diseases (e.g. overall risk profile of Afar region)
afar_overall_status = query_engine.query_region_all_diseases(region="Afar")

# Workflow 3: District-Wise Single Disease (e.g. high-resolution Anthrax risk for Jinka woreda)
jinka_anthrax_risk = query_engine.query_district_single_disease(
    district="Jinka", 
    disease_name="Anthrax"
)

print(f"[QUERY 1] FMD risk records in South Ethiopia: {len(south_ethiopia_fmd)}")
print(f"[QUERY 2] Districts mapped in Afar general report: {list(afar_overall_status.keys())}")
if jinka_anthrax_risk:
    print(f"[QUERY 3] Jinka Anthrax Risk Probability: {jinka_anthrax_risk['risk_probability'] * 100}% -> ({jinka_anthrax_risk['risk_class']})")

## Step 4: Export GIS and Dashboard Payloads
Using the `DashboardVisualizationEngine`, we compile spatial data structures to render maps and chart elements.

In [ ]:
from agents.dashboard_visualization_engine import DashboardVisualizationEngine

vis_engine = DashboardVisualizationEngine(df_risk)

# 1. Export Web Chart JSON metadata payload
chart_payload = vis_engine.compile_analytical_dashboard_payload()

# 2. Export Heatmap Coordinate Arrays (Leaflet.js mapping format)
anthrax_heat_coords = vis_engine.compile_risk_map_arrays(target_disease="Anthrax")

# 3. Export standard GeoJSON Point Feature Collection
geojson_collection = vis_engine.compile_geojson_collection()

# Save GeoJSON to our assets folder
geojson_output_path = "../docs/risk_map_example.json"
with open(geojson_output_path, "w") as f:
    json.dump(geojson_collection, f, indent=2)

print(f"[SUCCESS] Spatial GeoJSON mapping assets successfully compiled and saved to: {geojson_output_path}")
print(f"Compiled {len(anthrax_heat_coords)} heat coordinates for active Anthrax cases.")

## Step 5: Test the Monthly Prediction Bulletin Library
Finally, we run our `PredictionBulletinEngine` to catalogue, index, and query historical Monthly Prediction Bulletin files.

In [ ]:
from agents.prediction_bulletin_engine import PredictionBulletinEngine

# Initialize Bulletin Library
library = PredictionBulletinEngine()

# Index a new 2-month-ahead bulletin generated by our Communication Agent
new_forecast = {
    "id": "rep_003",
    "title": "August 2026 prediction report of October 2026 - Part 1",
    "publish_date": "2026-08-09",
    "forecast_target_month": "October 2026",
    "category": "Prediction",
    "is_latest": True,
    "file_path": "../docs/bulletins/bulletin_2026_08.md"
}
library.add_new_report(new_forecast)

# Execute search query mimicking our UI header filters: Year = 2026, Category = 'Prediction'
results = library.query_bulletin_library(search_query="Part 1", year_filter=2026)

print(f"\nSearch completed. Found {len(results)} matching reports inside our library catalog:")
for r in results:
    print(f"  - Title: {r['title']} | Published: {r['publish_date']}")